In [1]:
# ============================================================
# CELL 1 - Setup, Paths, Constants, and Data Contract
# ============================================================

!pip install neurokit2 -q

import json
import os
import pickle
import random
from datetime import datetime, timezone

import neurokit2 as nk
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import signal
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

ROOT = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS'
BASE = os.path.join(ROOT, 'Outputs- New')
WESAD_RAW_DIR = os.path.join(ROOT, 'Datasets', 'WESAD')
WESAD_DIR = os.path.join(BASE, 'WESAD')
AROAD_RAW_DIR = os.path.join(ROOT, 'Datasets', 'AffectiveROAD')
AROAD_BIO_DIR = os.path.join(AROAD_RAW_DIR, 'Bioharness')
AROAD_E4_DIR = os.path.join(AROAD_RAW_DIR, 'E4')

AE_DIR = os.path.join(BASE, 'LSTM_AE')
AE_CHECKPOINT_DIR = os.path.join(AE_DIR, 'checkpoints')
AE_MODELS_DIR = os.path.join(AE_DIR, 'models')
AE_LOSO_MODELS_DIR = os.path.join(AE_MODELS_DIR, 'loso')

# Forecasting artifacts are kept separate from the EDA and autoencoder outputs.
FORECAST_DIR = os.path.join(BASE, 'DIRECT_FORECASTER')
FORECAST_MODELS_DIR = os.path.join(FORECAST_DIR, 'models')
FORECAST_LOSO_MODELS_DIR = os.path.join(FORECAST_MODELS_DIR, 'loso')
FORECAST_RESULTS_DIR = os.path.join(FORECAST_DIR, 'results')
FORECAST_CHECKPOINT_DIR = os.path.join(FORECAST_DIR, 'checkpoints')

for folder in [
    FORECAST_MODELS_DIR,
    FORECAST_LOSO_MODELS_DIR,
    FORECAST_RESULTS_DIR,
    FORECAST_CHECKPOINT_DIR,
]:
    os.makedirs(folder, exist_ok=True)

FEATURE_NAMES = [
    'mean_HR', 'mean_RR', 'SDNN', 'RMSSD',
    'mean_BR', 'std_BR',
    'mean_temp', 'std_temp',
    'mean_acc_mag', 'std_acc_mag',
]
N_FEATURES = len(FEATURE_NAMES)
AE_SEQUENCE_LENGTH = 5
AE_HIDDEN_SIZE = 64

# Two distinct five-minute history blocks -> direct +5 and +10 minute targets.
HISTORY_BLOCKS = 2
FORECAST_BLOCKS = 2
BLOCK_MINUTES = 5
REPORTED_HORIZONS_MINUTES = [5, 10]
MINIMUM_CONTINUOUS_MINUTES = (
    HISTORY_BLOCKS + FORECAST_BLOCKS
) * BLOCK_MINUTES

SAMPLING_RATE = 700
WINDOW_SECONDS = 60
STEP_SECONDS = 60
PRE_STRESS_MINUTES = 40
REBUILD_TIMELINES = False
REBUILD_AR_TIMELINES = False
SEED = 42
SCORE_EPS = 1e-6

# Fixed before outer testing. Alpha is selected only by inner participant-wise CV.
ALPHA_GRID = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]
MIN_EVALUATED_SUBJECTS = 10
BOOTSTRAP_REPEATS = 10000
MIN_EXTERNAL_DRIVES = 10
MIN_EXTERNAL_ORIGINS = 30

AE_MODEL_VERSION = 'c1-unmasked-lstm-ae-wesad-v2'
FORECAST_MODEL_VERSION = 'c1-direct-ridge-score-forecast-wesad-v5'

WESAD_SUBJECTS = [
    'S2', 'S4', 'S5', 'S7', 'S8', 'S9', 'S10',
    'S11', 'S13', 'S14', 'S15', 'S16', 'S17',
]
AROAD_DRIVES = [f'Drv{i}' for i in range(1, 14)]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

required_files = [
    os.path.join(AE_MODELS_DIR, 'LSTM_AE_FINAL.pth'),
    os.path.join(AE_MODELS_DIR, 'LSTM_AE_FINAL_metadata.json'),
]
required_files.extend([
    os.path.join(AE_LOSO_MODELS_DIR, f'LSTM_AE_LOSO_{sid}.pth')
    for sid in WESAD_SUBJECTS
])
required_files.extend([
    os.path.join(AE_CHECKPOINT_DIR, f'WESAD_{sid}_processed.npz')
    for sid in WESAD_SUBJECTS
])
missing = [path for path in required_files if not os.path.exists(path)]
assert not missing, 'Run the new unmasked AE notebook first. Missing:\n' + '\n'.join(missing)

with open(os.path.join(AE_MODELS_DIR, 'LSTM_AE_FINAL_metadata.json')) as file:
    ae_metadata = json.load(file)

assert ae_metadata['model_version'] == AE_MODEL_VERSION
assert ae_metadata['predict_score_formula'] == (
    'raw_error / (raw_error + baseline_p95_raw_error)'
)

print(f'Device: {device}')
print(f'Frozen AE version: {AE_MODEL_VERSION}')
print(f'Forecast model version: {FORECAST_MODEL_VERSION}')
print(f'History: {HISTORY_BLOCKS} distinct five-minute blocks ({HISTORY_BLOCKS * 5} min)')
print('Targets: next distinct block at +5 min and following block at +10 min')
print(f'Minimum uninterrupted span for one evaluation origin: {MINIMUM_CONTINUOUS_MINUTES} min')
print(f'Participants: {len(WESAD_SUBJECTS)}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 9.0 MB/s eta 0:00:00
Mounted at /content/drive
Device: cpu
Frozen AE version: c1-unmasked-lstm-ae-wesad-v2
Forecast model version: c1-direct-ridge-score-forecast-wesad-v5
History: 2 distinct five-minute blocks (10 min)
Targets: next distinct block at +5 min and following block at +10 min
Minimum uninterrupted span for one evaluation origin: 20 min
Participants: 13


In [2]:
# ============================================================
# CELL 2 - Build or Reuse True Continuous WESAD Minute Timelines
# ============================================================


def extract_features_from_window(
    ecg_window,
    resp_window,
    temp_window,
    acc_window,
    sampling_rate=SAMPLING_RATE,
):
    """Exact ten-feature logic retained from the corrected WESAD preparation."""
    features = []

    try:
        ecg_cleaned = nk.ecg_clean(ecg_window, sampling_rate=sampling_rate)
        peaks, _ = nk.ecg_peaks(ecg_cleaned, sampling_rate=sampling_rate)
        peak_idx = np.where(peaks['ECG_R_Peaks'] == 1)[0]
        if len(peak_idx) < 31:
            return None

        rr_ms = np.diff(peak_idx) / sampling_rate * 1000
        rr_ms = rr_ms[(rr_ms >= 300) & (rr_ms <= 2000)]
        if len(rr_ms) < 30:
            return None

        mean_rr = float(np.mean(rr_ms))
        mean_hr = 60000.0 / mean_rr
        sdnn = float(np.std(rr_ms, ddof=1))
        rmssd = float(np.sqrt(np.mean(np.diff(rr_ms) ** 2)))
        if not np.all(np.isfinite([mean_hr, mean_rr, sdnn, rmssd])):
            return None
        features.extend([mean_hr, mean_rr, sdnn, rmssd])
    except Exception:
        return None

    try:
        resp_cleaned = nk.rsp_clean(resp_window, sampling_rate=sampling_rate)
        rsp_rate_arr = nk.rsp_rate(resp_cleaned, sampling_rate=sampling_rate)
        valid_br = rsp_rate_arr[(rsp_rate_arr >= 6) & (rsp_rate_arr <= 40)]
        if len(valid_br) < 0.8 * len(rsp_rate_arr):
            return None
        mean_br = float(np.mean(valid_br))
        std_br = float(np.std(valid_br, ddof=1)) if len(valid_br) > 1 else 0.0
        if not np.all(np.isfinite([mean_br, std_br])):
            return None
        features.extend([mean_br, std_br])
    except Exception:
        return None

    valid_temp = temp_window[(temp_window >= 25) & (temp_window <= 40)]
    if len(valid_temp) < 0.8 * len(temp_window):
        return None
    mean_temp = float(np.mean(valid_temp))
    std_temp = float(np.std(valid_temp, ddof=1)) if len(valid_temp) > 1 else 0.0
    if not np.all(np.isfinite([mean_temp, std_temp])):
        return None
    features.extend([mean_temp, std_temp])

    mean_acc_mag = float(np.mean(acc_window))
    std_acc_mag = float(np.std(acc_window, ddof=1))
    if not np.all(np.isfinite([mean_acc_mag, std_acc_mag])):
        return None
    features.extend([mean_acc_mag, std_acc_mag])

    return np.asarray(features, dtype=np.float32)


def valid_signal_window(ecg, resp, temp, acc):
    arrays = [ecg, resp, temp, acc]
    if any(not np.all(np.isfinite(values)) for values in arrays):
        return False
    if np.std(ecg) < 1e-6 or np.std(resp) < 1e-6 or np.std(temp) < 1e-6:
        return False
    if np.min(temp) < 10 or np.max(acc) > 1.5:
        return False
    return True


def majority_protocol_label(label_window):
    values, counts = np.unique(label_window.astype(int), return_counts=True)
    return int(values[np.argmax(counts)])


def prepare_wesad_forecast_timeline(sid):
    """Create minute-aligned features before and through the stress condition."""
    pkl_path = os.path.join(WESAD_RAW_DIR, sid, f'{sid}.pkl')
    mean_path = os.path.join(WESAD_DIR, 'per_subject', f'{sid}_norm_params_mean.npy')
    std_path = os.path.join(WESAD_DIR, 'per_subject', f'{sid}_norm_params_std.npy')
    for path in [pkl_path, mean_path, std_path]:
        assert os.path.exists(path), f'Missing required WESAD file: {path}'

    with open(pkl_path, 'rb') as file:
        data = pickle.load(file, encoding='latin1')

    chest = data['signal']['chest']
    labels = data['label'].flatten().astype(int)
    ecg = chest['ECG'].flatten()
    resp = chest['Resp'].flatten()
    temp = chest['Temp'].flatten()
    acc = chest['ACC']

    b, a = signal.butter(4, 0.5 / (0.5 * SAMPLING_RATE), btype='high')
    acc_filtered = np.column_stack([
        signal.filtfilt(b, a, acc[:, axis]) for axis in range(3)
    ])
    acc_mag = np.sqrt(np.sum(acc_filtered ** 2, axis=1))

    stress_idx = np.where(labels == 2)[0]
    assert len(stress_idx) > 0, f'{sid}: stress label is missing'
    stress_start = int(stress_idx[0])
    stress_end = int(stress_idx[-1]) + 1

    timeline_start = max(
        0,
        stress_start - PRE_STRESS_MINUTES * 60 * SAMPLING_RATE,
    )
    timeline_end = stress_end
    window_size = WINDOW_SECONDS * SAMPLING_RATE
    step_size = STEP_SECONDS * SAMPLING_RATE

    baseline_mean = np.load(mean_path).astype(np.float32)
    baseline_std = np.load(std_path).astype(np.float32)
    baseline_std = np.where(baseline_std == 0, 1e-8, baseline_std)

    features = []
    starts = []
    protocol_labels = []
    minutes_from_stress = []
    rejected = 0

    for start in range(timeline_start, timeline_end - window_size + 1, step_size):
        end = start + window_size
        ecg_window = ecg[start:end]
        resp_window = resp[start:end]
        temp_window = temp[start:end]
        acc_window = acc_mag[start:end]

        if not valid_signal_window(ecg_window, resp_window, temp_window, acc_window):
            rejected += 1
            continue

        feature_vector = extract_features_from_window(
            ecg_window,
            resp_window,
            temp_window,
            acc_window,
        )
        if feature_vector is None:
            rejected += 1
            continue

        normalized = (feature_vector - baseline_mean) / baseline_std
        if not np.all(np.isfinite(normalized)):
            rejected += 1
            continue

        features.append(normalized.astype(np.float32))
        starts.append(float(start / SAMPLING_RATE))
        protocol_labels.append(majority_protocol_label(labels[start:end]))
        minutes_from_stress.append(float((start - stress_start) / SAMPLING_RATE / 60.0))

    output_path = os.path.join(
        FORECAST_CHECKPOINT_DIR,
        f'WESAD_{sid}_forecast_timeline.npz',
    )
    np.savez_compressed(
        output_path,
        features=np.asarray(features, dtype=np.float32),
        start_sec=np.asarray(starts, dtype=np.float64),
        protocol_labels=np.asarray(protocol_labels, dtype=np.int16),
        minutes_from_stress=np.asarray(minutes_from_stress, dtype=np.float32),
    )
    return len(features), rejected, output_path


print('=' * 60)
print('CONTINUOUS WESAD TIMELINES')
print('=' * 60)

timeline_summary_path = os.path.join(
    FORECAST_RESULTS_DIR,
    'timeline_preparation_summary.json',
)
timeline_summary = {}

for sid in WESAD_SUBJECTS:
    new_path = os.path.join(
        FORECAST_CHECKPOINT_DIR,
        f'WESAD_{sid}_forecast_timeline.npz',
    )
    # A cached timeline contains features only, never predictions or fitted values.
    if os.path.exists(new_path) and not REBUILD_TIMELINES:
        existing = np.load(new_path)
        features = existing['features'].astype(np.float32)
        starts = existing['start_sec'].astype(np.float64)
        protocol_labels = existing['protocol_labels'].astype(np.int16)
        minutes_from_stress = existing['minutes_from_stress'].astype(np.float32)

        assert features.ndim == 2 and features.shape[1] == N_FEATURES
        assert len(features) == len(starts) == len(protocol_labels) == len(minutes_from_stress)
        assert np.isfinite(features).all() and np.isfinite(starts).all()

        n_valid = len(features)
        rejected = None
        source = 'reused exact feature timeline'
        output_path = new_path
    else:
        n_valid, rejected, output_path = prepare_wesad_forecast_timeline(sid)
        source = 'rebuilt from raw WESAD'

    timeline_summary[sid] = {
        'valid_minutes': int(n_valid),
        'rejected_minutes': int(rejected) if rejected is not None else None,
        'file': output_path,
        'source': source,
    }
    print(f'{sid}: valid minutes={n_valid}, source={source}')

with open(timeline_summary_path, 'w') as file:
    json.dump(timeline_summary, file, indent=2)

print('The WESAD EDA outputs were not modified.')


CONTINUOUS WESAD TIMELINES
S2: valid minutes=47, source=reused exact feature timeline
S4: valid minutes=49, source=reused exact feature timeline
S5: valid minutes=48, source=reused exact feature timeline
S7: valid minutes=47, source=reused exact feature timeline
S8: valid minutes=45, source=reused exact feature timeline
S9: valid minutes=39, source=reused exact feature timeline
S10: valid minutes=49, source=reused exact feature timeline
S11: valid minutes=42, source=reused exact feature timeline
S13: valid minutes=50, source=reused exact feature timeline
S14: valid minutes=40, source=reused exact feature timeline
S15: valid minutes=48, source=reused exact feature timeline
S16: valid minutes=43, source=reused exact feature timeline
S17: valid minutes=46, source=reused exact feature timeline
The WESAD EDA outputs were not modified.


In [3]:
# ============================================================
# CELL 3 - Frozen AE, Training-Only Calibration, and Block Builder
# ============================================================


class LSTMAutoEncoder(nn.Module):
    """Exact architecture from the accepted unmasked AE notebook."""

    def __init__(self, n_features=N_FEATURES, hidden_size=AE_HIDDEN_SIZE, n_layers=1):
        super().__init__()
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
        )
        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
        )
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        bottleneck = h_n[-1]
        decoder_input = bottleneck.unsqueeze(1).repeat(1, AE_SEQUENCE_LENGTH, 1)
        decoder_output, _ = self.decoder(decoder_input)
        return self.output_layer(decoder_output)


def load_frozen_ae(model_path):
    assert os.path.exists(model_path), f'Missing AE model: {model_path}'
    model = LSTMAutoEncoder().to(device)
    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.eval()
    return model


def score_ae_sequences(model, sequences, batch_size=256):
    sequences = np.asarray(sequences, dtype=np.float32)
    if len(sequences) == 0:
        return np.empty(0, dtype=np.float64)

    outputs = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(sequences), batch_size):
            batch = torch.tensor(
                sequences[start:start + batch_size],
                dtype=torch.float32,
                device=device,
            )
            reconstruction = model(batch)
            errors = torch.mean((batch - reconstruction) ** 2, dim=(1, 2))
            outputs.append(errors.cpu().numpy())
    return np.concatenate(outputs).astype(np.float64)


def training_only_p95(ae_model, training_subjects):
    """
    Match the accepted C1 calibration rule without touching the held-out person.

    The AE notebook defines its deployed p95 on its baseline training sequences.
    We repeat that exact calibration contract for each outer fold using only that
    fold's training participants.
    """
    baseline_error_parts = []
    for sid in training_subjects:
        data = np.load(os.path.join(AE_CHECKPOINT_DIR, f'WESAD_{sid}_processed.npz'))
        baseline_error_parts.append(
            score_ae_sequences(ae_model, data['base_seqs'].astype(np.float32))
        )

    baseline_errors = np.concatenate(baseline_error_parts)
    assert len(baseline_errors) > 0 and np.isfinite(baseline_errors).all()
    p95 = float(np.percentile(baseline_errors, 95))
    assert p95 > 0
    return p95


def to_c1_score(raw_error, baseline_p95):
    """Existing deployed C1 score definition; it is not a probability."""
    raw_error = np.asarray(raw_error, dtype=np.float64)
    score = raw_error / (raw_error + float(baseline_p95) + 1e-12)
    return np.clip(score, 0.0, 1.0)


def score_to_logit(score):
    """Bounded coordinate transform only; C1 is still not interpreted as probability."""
    score = np.clip(np.asarray(score, dtype=np.float64), SCORE_EPS, 1.0 - SCORE_EPS)
    return np.log(score) - np.log1p(-score)


def logit_to_score(value):
    value = np.asarray(value, dtype=np.float64)
    positive = value >= 0
    output = np.empty_like(value, dtype=np.float64)
    output[positive] = 1.0 / (1.0 + np.exp(-value[positive]))
    exp_value = np.exp(value[~positive])
    output[~positive] = exp_value / (1.0 + exp_value)
    return output


def contiguous_slices(starts, expected_step=STEP_SECONDS):
    """Split at every missing minute; no block or sample can cross a gap."""
    starts = np.asarray(starts, dtype=np.float64)
    if len(starts) == 0:
        return []
    breaks = np.where(
        ~np.isclose(np.diff(starts), expected_step, rtol=0.0, atol=1e-6)
    )[0] + 1
    boundaries = np.concatenate([[0], breaks, [len(starts)]])
    return [
        slice(boundaries[index], boundaries[index + 1])
        for index in range(len(boundaries) - 1)
    ]


def build_distinct_ae_blocks(feature_run, start_run):
    """
    Make adjacent five-minute blocks with stride five minutes.

    Block 0 uses minutes 0-4 and block 1 uses minutes 5-9. No raw minute
    appears in two AE blocks.
    """
    feature_run = np.asarray(feature_run, dtype=np.float32)
    start_run = np.asarray(start_run, dtype=np.float64)
    n_blocks = len(feature_run) // AE_SEQUENCE_LENGTH
    if n_blocks == 0:
        return (
            np.empty((0, AE_SEQUENCE_LENGTH, N_FEATURES), dtype=np.float32),
            np.empty(0, dtype=np.float64),
        )

    blocks = []
    block_starts = []
    for block_index in range(n_blocks):
        first = block_index * AE_SEQUENCE_LENGTH
        last = first + AE_SEQUENCE_LENGTH
        block_starts_raw = start_run[first:last]
        assert np.allclose(
            np.diff(block_starts_raw),
            STEP_SECONDS,
            rtol=0.0,
            atol=1e-6,
        )
        blocks.append(feature_run[first:last])
        block_starts.append(float(block_starts_raw[0]))

    blocks = np.asarray(blocks, dtype=np.float32)
    block_starts = np.asarray(block_starts, dtype=np.float64)
    if len(block_starts) > 1:
        assert np.allclose(
            np.diff(block_starts),
            BLOCK_MINUTES * 60,
            rtol=0.0,
            atol=1e-6,
        )
    return blocks, block_starts


def build_rolling_origins(block_scores, block_starts):
    """Build two-block history and the next +5/+10 minute block targets."""
    block_scores = np.asarray(block_scores, dtype=np.float64)
    block_starts = np.asarray(block_starts, dtype=np.float64)
    required_blocks = HISTORY_BLOCKS + FORECAST_BLOCKS
    if len(block_scores) < required_blocks:
        return (
            np.empty((0, HISTORY_BLOCKS), dtype=np.float64),
            np.empty((0, FORECAST_BLOCKS), dtype=np.float64),
            np.empty(0, dtype=np.float64),
            np.empty(0, dtype=np.float64),
        )

    X, y, persistence, origin_end = [], [], [], []
    for origin in range(len(block_scores) - required_blocks + 1):
        history_end = origin + HISTORY_BLOCKS
        target_end = history_end + FORECAST_BLOCKS
        X.append(block_scores[origin:history_end])
        y.append(block_scores[history_end:target_end])
        persistence.append(block_scores[history_end - 1])
        origin_end.append(
            block_starts[history_end - 1] + BLOCK_MINUTES * 60
        )

    return (
        np.asarray(X, dtype=np.float64),
        np.asarray(y, dtype=np.float64),
        np.asarray(persistence, dtype=np.float64),
        np.asarray(origin_end, dtype=np.float64),
    )


def timeline_to_samples(features, starts, ae_model, baseline_p95):
    """Apply all block and forecast construction inside uninterrupted runs."""
    X_parts, y_parts, persistence_parts, origin_parts = [], [], [], []
    raw_block_errors = []
    n_blocks_total = 0

    for run_slice in contiguous_slices(starts):
        blocks, block_starts = build_distinct_ae_blocks(
            features[run_slice],
            starts[run_slice],
        )
        if len(blocks) == 0:
            continue

        errors = score_ae_sequences(ae_model, blocks)
        scores = to_c1_score(errors, baseline_p95)
        X_run, y_run, persistence_run, origin_run = build_rolling_origins(
            scores,
            block_starts,
        )
        n_blocks_total += len(blocks)
        raw_block_errors.append(errors)

        if len(X_run) > 0:
            X_parts.append(X_run)
            y_parts.append(y_run)
            persistence_parts.append(persistence_run)
            origin_parts.append(origin_run)

    if not X_parts:
        return {
            'X': np.empty((0, HISTORY_BLOCKS), dtype=np.float64),
            'y': np.empty((0, FORECAST_BLOCKS), dtype=np.float64),
            'persistence': np.empty(0, dtype=np.float64),
            'origin_end_sec': np.empty(0, dtype=np.float64),
            'raw_block_errors': (
                np.concatenate(raw_block_errors)
                if raw_block_errors else np.empty(0, dtype=np.float64)
            ),
            'n_distinct_blocks': int(n_blocks_total),
        }

    return {
        'X': np.concatenate(X_parts, axis=0),
        'y': np.concatenate(y_parts, axis=0),
        'persistence': np.concatenate(persistence_parts, axis=0),
        'origin_end_sec': np.concatenate(origin_parts, axis=0),
        'raw_block_errors': np.concatenate(raw_block_errors),
        'n_distinct_blocks': int(n_blocks_total),
    }


def wesad_subject_samples(sid, ae_model, baseline_p95):
    timeline_path = os.path.join(
        FORECAST_CHECKPOINT_DIR,
        f'WESAD_{sid}_forecast_timeline.npz',
    )
    timeline = np.load(timeline_path)
    return timeline_to_samples(
        timeline['features'].astype(np.float32),
        timeline['start_sec'].astype(np.float64),
        ae_model,
        baseline_p95,
    )


# Data audit uses the final AE only to count valid construction paths.
final_ae_for_audit = load_frozen_ae(os.path.join(AE_MODELS_DIR, 'LSTM_AE_FINAL.pth'))
final_p95_for_audit = float(ae_metadata['baseline_p95_raw_error'])
sample_count_audit = {}

print('=' * 60)
print('NON-OVERLAPPING FIVE-MINUTE BLOCK AUDIT')
print('=' * 60)
for sid in WESAD_SUBJECTS:
    subject_data = wesad_subject_samples(sid, final_ae_for_audit, final_p95_for_audit)
    sample_count_audit[sid] = int(len(subject_data['X']))
    print(
        f'{sid}: distinct blocks={subject_data["n_distinct_blocks"]}, '
        f'rolling forecast origins={len(subject_data["X"])}'
    )

del final_ae_for_audit
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    'Participants with at least one honest 20-minute origin: '
    f'{sum(count > 0 for count in sample_count_audit.values())}/{len(WESAD_SUBJECTS)}'
)
print('No gaps were joined and no raw minute was used in two AE blocks.')


NON-OVERLAPPING FIVE-MINUTE BLOCK AUDIT
S2: distinct blocks=9, rolling forecast origins=5
S4: distinct blocks=9, rolling forecast origins=4
S5: distinct blocks=9, rolling forecast origins=2
S7: distinct blocks=8, rolling forecast origins=2
S8: distinct blocks=8, rolling forecast origins=2
S9: distinct blocks=7, rolling forecast origins=4
S10: distinct blocks=9, rolling forecast origins=3
S11: distinct blocks=8, rolling forecast origins=5
S13: distinct blocks=9, rolling forecast origins=6
S14: distinct blocks=7, rolling forecast origins=3
S15: distinct blocks=8, rolling forecast origins=1
S16: distinct blocks=8, rolling forecast origins=5
S17: distinct blocks=7, rolling forecast origins=1
Participants with at least one honest 20-minute origin: 13/13
No gaps were joined and no raw minute was used in two AE blocks.


In [4]:
# ============================================================
# CELL 4 - Direct Residual Ridge Forecaster and Nested-CV Helpers
# ============================================================


def equal_subject_weights(subject_ids):
    """Give every participant total weight 1, regardless of origin count."""
    subject_ids = np.asarray(subject_ids).astype(str)
    unique, counts = np.unique(subject_ids, return_counts=True)
    count_map = dict(zip(unique, counts))
    weights = np.asarray(
        [1.0 / count_map[sid] for sid in subject_ids],
        dtype=np.float64,
    )
    # Mean 1 keeps alpha magnitudes easier to interpret across folds.
    weights *= len(weights) / np.sum(weights)
    return weights


def combine_subject_data(data_by_subject, subject_list):
    usable = [sid for sid in subject_list if len(data_by_subject[sid]['X']) > 0]
    if not usable:
        return (
            np.empty((0, HISTORY_BLOCKS), dtype=np.float64),
            np.empty((0, FORECAST_BLOCKS), dtype=np.float64),
            np.empty(0, dtype=str),
        )
    X = np.concatenate([data_by_subject[sid]['X'] for sid in usable], axis=0)
    y = np.concatenate([data_by_subject[sid]['y'] for sid in usable], axis=0)
    subject_ids = np.concatenate([
        np.repeat(sid, len(data_by_subject[sid]['X'])) for sid in usable
    ])
    return X, y, subject_ids


def fit_direct_ridge(X_score, y_score, subject_ids, alpha):
    """
    Learn corrections to persistence in bounded-score logit coordinates.

    Predicting a correction, rather than an unanchored absolute raw error, makes
    zero correction exactly equal to the required persistence benchmark. This is
    learned during fitting; there is no manual adjustment after prediction.
    """
    assert len(X_score) == len(y_score) == len(subject_ids) and len(X_score) > 0
    X_logit = score_to_logit(X_score)
    y_logit = score_to_logit(y_score)
    target_delta = y_logit - X_logit[:, [-1]]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_logit)
    weights = equal_subject_weights(subject_ids)

    ridge = Ridge(alpha=float(alpha), fit_intercept=True)
    ridge.fit(X_scaled, target_delta, sample_weight=weights)

    return {
        'scaler': scaler,
        'ridge': ridge,
        'alpha': float(alpha),
    }


def predict_direct_ridge(fitted, X_score):
    X_score = np.asarray(X_score, dtype=np.float64)
    if len(X_score) == 0:
        return np.empty((0, FORECAST_BLOCKS), dtype=np.float64)
    X_logit = score_to_logit(X_score)
    X_scaled = fitted['scaler'].transform(X_logit)
    predicted_delta = fitted['ridge'].predict(X_scaled)
    predicted_future_logit = X_logit[:, [-1]] + predicted_delta
    return logit_to_score(predicted_future_logit)


def select_alpha_inner_loso(data_by_subject, candidate_subjects):
    """
    Tune alpha without the outer test participant.

    The one-standard-error rule chooses the strongest regularization whose inner
    participant-level MAE is statistically indistinguishable from the best mean.
    This reduces selection noise in a small dataset.
    """
    usable = [sid for sid in candidate_subjects if len(data_by_subject[sid]['X']) > 0]
    assert len(usable) >= 3, 'Fewer than three training participants have samples.'

    alpha_details = {}
    for alpha in ALPHA_GRID:
        heldout_subject_mae = []
        for validation_sid in usable:
            inner_train_subjects = [sid for sid in usable if sid != validation_sid]
            X_train, y_train, train_ids = combine_subject_data(
                data_by_subject,
                inner_train_subjects,
            )
            fitted = fit_direct_ridge(X_train, y_train, train_ids, alpha)
            validation = data_by_subject[validation_sid]
            predictions = predict_direct_ridge(fitted, validation['X'])
            heldout_subject_mae.append(
                float(np.mean(np.abs(predictions - validation['y'])))
            )

        values = np.asarray(heldout_subject_mae, dtype=np.float64)
        standard_error = (
            float(np.std(values, ddof=1) / np.sqrt(len(values)))
            if len(values) > 1 else 0.0
        )
        alpha_details[str(alpha)] = {
            'mean_subject_mae': float(np.mean(values)),
            'standard_error': standard_error,
            'n_validation_subjects': int(len(values)),
            'subject_mae': [float(value) for value in values],
        }

    best_alpha = min(
        ALPHA_GRID,
        key=lambda alpha: alpha_details[str(alpha)]['mean_subject_mae'],
    )
    best_mean = alpha_details[str(best_alpha)]['mean_subject_mae']
    best_se = alpha_details[str(best_alpha)]['standard_error']
    eligible = [
        alpha for alpha in ALPHA_GRID
        if alpha_details[str(alpha)]['mean_subject_mae'] <= best_mean + best_se
    ]
    selected_alpha = float(max(eligible))

    return selected_alpha, {
        'selection_rule': 'largest alpha within one standard error of best inner mean MAE',
        'unregularized_best_alpha': float(best_alpha),
        'selected_alpha': selected_alpha,
        'candidate_results': alpha_details,
    }


def horizon_metrics(predictions, targets, persistence):
    predictions = np.asarray(predictions, dtype=np.float64)
    targets = np.asarray(targets, dtype=np.float64)
    persistence = np.asarray(persistence, dtype=np.float64)
    assert predictions.shape == targets.shape
    assert len(predictions) == len(persistence) and len(predictions) > 0

    result = {}
    for horizon_index, horizon_minutes in enumerate(REPORTED_HORIZONS_MINUTES):
        model_errors = np.abs(predictions[:, horizon_index] - targets[:, horizon_index])
        persistence_errors = np.abs(
            persistence - targets[:, horizon_index]
        )
        result[f'MAE_plus_{horizon_minutes}m'] = float(np.mean(model_errors))
        result[f'persistence_MAE_plus_{horizon_minutes}m'] = float(
            np.mean(persistence_errors)
        )
        result[f'MAE_improvement_over_persistence_plus_{horizon_minutes}m'] = float(
            np.mean(persistence_errors) - np.mean(model_errors)
        )
    result['MAE_both_horizons'] = float(np.mean(np.abs(predictions - targets)))
    return result


class DirectScoreForecaster(nn.Module):
    """Tiny inference module matching the fitted residual Ridge equation."""

    def __init__(self):
        super().__init__()
        self.register_buffer('input_mean', torch.zeros(HISTORY_BLOCKS))
        self.register_buffer('input_scale', torch.ones(HISTORY_BLOCKS))
        self.delta_layer = nn.Linear(HISTORY_BLOCKS, FORECAST_BLOCKS)

    def forward(self, score_history):
        bounded = torch.clamp(score_history, SCORE_EPS, 1.0 - SCORE_EPS)
        history_logit = torch.log(bounded) - torch.log1p(-bounded)
        scaled = (history_logit - self.input_mean) / self.input_scale
        predicted_delta = self.delta_layer(scaled)
        future_logit = history_logit[:, -1:] + predicted_delta
        return torch.sigmoid(future_logit)


def fitted_ridge_to_torch(fitted):
    model = DirectScoreForecaster()
    with torch.no_grad():
        model.input_mean.copy_(
            torch.tensor(fitted['scaler'].mean_, dtype=torch.float32)
        )
        model.input_scale.copy_(
            torch.tensor(fitted['scaler'].scale_, dtype=torch.float32)
        )
        model.delta_layer.weight.copy_(
            torch.tensor(fitted['ridge'].coef_, dtype=torch.float32)
        )
        model.delta_layer.bias.copy_(
            torch.tensor(fitted['ridge'].intercept_, dtype=torch.float32)
        )
    model.eval()
    return model


smoke_X = np.asarray([[0.2, 0.3], [0.4, 0.5], [0.3, 0.35]], dtype=np.float64)
smoke_y = np.asarray([[0.35, 0.4], [0.55, 0.6], [0.4, 0.45]], dtype=np.float64)
smoke_ids = np.asarray(['A', 'B', 'C'])
smoke_fit = fit_direct_ridge(smoke_X, smoke_y, smoke_ids, alpha=1.0)
smoke_sklearn = predict_direct_ridge(smoke_fit, smoke_X)
smoke_torch_model = fitted_ridge_to_torch(smoke_fit)
with torch.no_grad():
    smoke_torch = smoke_torch_model(
        torch.tensor(smoke_X, dtype=torch.float32)
    ).numpy()
assert smoke_sklearn.shape == (3, 2)
assert np.all((smoke_sklearn > 0) & (smoke_sklearn < 1))
assert np.allclose(smoke_sklearn, smoke_torch, rtol=1e-5, atol=1e-6)
print('Direct forecaster check passed: 2 inputs, 2 bounded direct outputs.')
print(
    'Trainable coefficients in final inference model: '
    f'{sum(parameter.numel() for parameter in smoke_torch_model.parameters())}'
)


Direct forecaster check passed: 2 inputs, 2 bounded direct outputs.
Trainable coefficients in final inference model: 6


In [5]:
# ============================================================
# CELL 5 - Strict Outer WESAD Leave-One-Subject-Out Evaluation
# ============================================================

print('=' * 60)
print('STRICT WESAD OUTER LOSO FORECAST EVALUATION')
print('Outer participant is absent from AE weights, p95, Ridge fitting, and alpha choice.')
print('=' * 60)

loso_results = {}
not_evaluable = {}
pooled_prediction_parts = []
pooled_target_parts = []
pooled_persistence_parts = []
pooled_subject_parts = []
pooled_origin_parts = []

for fold_index, test_sid in enumerate(WESAD_SUBJECTS):
    outer_training_subjects = [sid for sid in WESAD_SUBJECTS if sid != test_sid]
    ae_path = os.path.join(AE_LOSO_MODELS_DIR, f'LSTM_AE_LOSO_{test_sid}.pth')
    fold_ae = load_frozen_ae(ae_path)

    # Critical: neither the held-out baseline nor its stress period sets this scale.
    fold_p95 = training_only_p95(fold_ae, outer_training_subjects)
    fold_data = {
        sid: wesad_subject_samples(sid, fold_ae, fold_p95)
        for sid in WESAD_SUBJECTS
    }

    test_data = fold_data[test_sid]
    if len(test_data['X']) == 0:
        not_evaluable[test_sid] = (
            f'no uninterrupted {MINIMUM_CONTINUOUS_MINUTES}-minute span after '
            'quality rejection; gaps were not joined'
        )
        print(f'{test_sid}: not evaluable; no honest 20-minute origin')
        del fold_ae
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

    selected_alpha, alpha_audit = select_alpha_inner_loso(
        fold_data,
        outer_training_subjects,
    )
    X_train, y_train, train_subject_ids = combine_subject_data(
        fold_data,
        outer_training_subjects,
    )
    fitted = fit_direct_ridge(
        X_train,
        y_train,
        train_subject_ids,
        selected_alpha,
    )
    predictions = predict_direct_ridge(fitted, test_data['X'])
    metrics = horizon_metrics(
        predictions,
        test_data['y'],
        test_data['persistence'],
    )

    training_counts = {
        sid: int(len(fold_data[sid]['X'])) for sid in outer_training_subjects
    }
    metrics.update({
        'test_subject': test_sid,
        'test_origins': int(len(test_data['X'])),
        'test_distinct_ae_blocks': int(test_data['n_distinct_blocks']),
        'train_origins': int(len(X_train)),
        'training_origins_by_subject': training_counts,
        'training_only_baseline_p95_raw_error': float(fold_p95),
        'selected_alpha': float(selected_alpha),
        'inner_alpha_selection': alpha_audit,
        'ae_model_file': os.path.basename(ae_path),
    })
    loso_results[test_sid] = metrics

    torch_model = fitted_ridge_to_torch(fitted)
    torch.save(
        torch_model.state_dict(),
        os.path.join(
            FORECAST_LOSO_MODELS_DIR,
            f'DIRECT_RIDGE_LOSO_{test_sid}.pth',
        ),
    )

    np.savez_compressed(
        os.path.join(
            FORECAST_RESULTS_DIR,
            f'WESAD_LOSO_{test_sid}_forecast_outputs.npz',
        ),
        predictions=predictions.astype(np.float32),
        targets=test_data['y'].astype(np.float32),
        persistence=test_data['persistence'].astype(np.float32),
        input_score_history=test_data['X'].astype(np.float32),
        origin_end_sec=test_data['origin_end_sec'].astype(np.float64),
    )

    pooled_prediction_parts.append(predictions)
    pooled_target_parts.append(test_data['y'])
    pooled_persistence_parts.append(test_data['persistence'])
    pooled_subject_parts.append(np.repeat(test_sid, len(predictions)))
    pooled_origin_parts.append(test_data['origin_end_sec'])

    print(
        f'{test_sid}: origins={len(predictions)}, alpha={selected_alpha:g}, '
        f'MAE +5={metrics["MAE_plus_5m"]:.4f} '
        f'(persistence {metrics["persistence_MAE_plus_5m"]:.4f}), '
        f'MAE +10={metrics["MAE_plus_10m"]:.4f} '
        f'(persistence {metrics["persistence_MAE_plus_10m"]:.4f})'
    )

    del fold_ae, fitted, torch_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert loso_results, 'No held-out participant had an honest forecast origin.'

pooled_predictions = np.concatenate(pooled_prediction_parts, axis=0)
pooled_targets = np.concatenate(pooled_target_parts, axis=0)
pooled_persistence = np.concatenate(pooled_persistence_parts, axis=0)
pooled_subject_ids = np.concatenate(pooled_subject_parts)
pooled_origin_end_sec = np.concatenate(pooled_origin_parts)

pooled_metrics = horizon_metrics(
    pooled_predictions,
    pooled_targets,
    pooled_persistence,
)

metric_names = [
    'MAE_plus_5m',
    'persistence_MAE_plus_5m',
    'MAE_improvement_over_persistence_plus_5m',
    'MAE_plus_10m',
    'persistence_MAE_plus_10m',
    'MAE_improvement_over_persistence_plus_10m',
    'MAE_both_horizons',
]
macro_metrics = {
    metric_name: float(np.mean([
        result[metric_name] for result in loso_results.values()
    ]))
    for metric_name in metric_names
}

np.savez_compressed(
    os.path.join(FORECAST_RESULTS_DIR, 'wesad_loso_oof_forecast_outputs.npz'),
    predictions=pooled_predictions.astype(np.float32),
    targets=pooled_targets.astype(np.float32),
    persistence=pooled_persistence.astype(np.float32),
    subject_ids=pooled_subject_ids,
    origin_end_sec=pooled_origin_end_sec.astype(np.float64),
    horizons_minutes=np.asarray(REPORTED_HORIZONS_MINUTES, dtype=np.int16),
)

print('\nPrimary participant-macro held-out results')
print(
    f'  +5 min MAE: model={macro_metrics["MAE_plus_5m"]:.6f}, '
    f'persistence={macro_metrics["persistence_MAE_plus_5m"]:.6f}, '
    f'improvement={macro_metrics["MAE_improvement_over_persistence_plus_5m"]:.6f}'
)
print(
    f'  +10 min MAE: model={macro_metrics["MAE_plus_10m"]:.6f}, '
    f'persistence={macro_metrics["persistence_MAE_plus_10m"]:.6f}, '
    f'improvement={macro_metrics["MAE_improvement_over_persistence_plus_10m"]:.6f}'
)
print('\nSecondary pooled-origin results (not treated as independent people)')
print(
    f'  +5 min improvement over persistence: '
    f'{pooled_metrics["MAE_improvement_over_persistence_plus_5m"]:.6f}'
)
print(
    f'  +10 min improvement over persistence: '
    f'{pooled_metrics["MAE_improvement_over_persistence_plus_10m"]:.6f}'
)


STRICT WESAD OUTER LOSO FORECAST EVALUATION
Outer participant is absent from AE weights, p95, Ridge fitting, and alpha choice.
S2: origins=5, alpha=100, MAE +5=0.2774 (persistence 0.2607), MAE +10=0.2001 (persistence 0.2838)
S4: origins=4, alpha=100, MAE +5=0.1285 (persistence 0.1311), MAE +10=0.0824 (persistence 0.0848)
S5: origins=2, alpha=100, MAE +5=0.0635 (persistence 0.0760), MAE +10=0.0649 (persistence 0.1236)
S7: origins=2, alpha=100, MAE +5=0.0750 (persistence 0.1039), MAE +10=0.0235 (persistence 0.0703)
S8: origins=2, alpha=100, MAE +5=0.1035 (persistence 0.0550), MAE +10=0.0860 (persistence 0.0909)
S9: origins=4, alpha=100, MAE +5=0.1602 (persistence 0.2148), MAE +10=0.1081 (persistence 0.2127)
S10: origins=3, alpha=100, MAE +5=0.0555 (persistence 0.0701), MAE +10=0.0437 (persistence 0.0443)
S11: origins=5, alpha=100, MAE +5=0.1218 (persistence 0.1302), MAE +10=0.1445 (persistence 0.1912)
S13: origins=6, alpha=100, MAE +5=0.0828 (persistence 0.1207), MAE +10=0.0723 (persiste

In [6]:
# ============================================================
# CELL 6 - Participant-Level Uncertainty and Deployment Gate
# ============================================================


def participant_bootstrap_ci(values, repeats=BOOTSTRAP_REPEATS, seed=SEED):
    """Resample participants, never individual rolling origins."""
    values = np.asarray(values, dtype=np.float64)
    assert len(values) > 1
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(values), size=(repeats, len(values)))
    bootstrap_means = np.mean(values[indices], axis=1)
    return {
        'mean': float(np.mean(values)),
        'lower_95': float(np.percentile(bootstrap_means, 2.5)),
        'upper_95': float(np.percentile(bootstrap_means, 97.5)),
        'unit': 'participant',
        'repeats': int(repeats),
    }


evaluated_subjects = list(loso_results.keys())
improvement_5 = np.asarray([
    loso_results[sid]['MAE_improvement_over_persistence_plus_5m']
    for sid in evaluated_subjects
])
improvement_10 = np.asarray([
    loso_results[sid]['MAE_improvement_over_persistence_plus_10m']
    for sid in evaluated_subjects
])

ci_5 = participant_bootstrap_ci(improvement_5, seed=SEED + 5)
ci_10 = participant_bootstrap_ci(improvement_10, seed=SEED + 10)
n_better_5 = int(np.sum(improvement_5 > 0))
n_better_10 = int(np.sum(improvement_10 > 0))
n_better_both = int(np.sum((improvement_5 > 0) & (improvement_10 > 0)))
strict_majority = len(evaluated_subjects) // 2 + 1

predictive_gate_checks = {
    'at_least_10_evaluable_subjects': bool(
        len(evaluated_subjects) >= MIN_EVALUATED_SUBJECTS
    ),
    'positive_macro_improvement_plus_5m': bool(np.mean(improvement_5) > 0),
    'positive_macro_improvement_plus_10m': bool(np.mean(improvement_10) > 0),
    'majority_of_subjects_better_at_both_horizons': bool(
        n_better_both >= strict_majority
    ),
}
uncertainty_checks = {
    'participant_bootstrap_95_lower_bound_above_zero_plus_5m': bool(
        ci_5['lower_95'] > 0
    ),
    'participant_bootstrap_95_lower_bound_above_zero_plus_10m': bool(
        ci_10['lower_95'] > 0
    ),
}
passes_internal_gate = bool(all(predictive_gate_checks.values()))

deployment_gate = {
    'model_version': FORECAST_MODEL_VERSION,
    'status': (
        'APPROVED_FOR_EXTERNAL_VALIDATION'
        if passes_internal_gate else 'NOT_APPROVED'
    ),
    'all_predictive_checks_required': True,
    'predictive_checks': predictive_gate_checks,
    'uncertainty_checks_reported_not_used_for_model_selection': uncertainty_checks,
    'evaluated_subjects': evaluated_subjects,
    'not_evaluable_subjects': not_evaluable,
    'n_evaluated_subjects': int(len(evaluated_subjects)),
    'strict_majority_required': int(strict_majority),
    'subjects_better_plus_5m': n_better_5,
    'subjects_better_plus_10m': n_better_10,
    'subjects_better_at_both_horizons': n_better_both,
    'participant_bootstrap_improvement_plus_5m': ci_5,
    'participant_bootstrap_improvement_plus_10m': ci_10,
}

with open(os.path.join(FORECAST_RESULTS_DIR, 'deployment_gate.json'), 'w') as file:
    json.dump(deployment_gate, file, indent=2)

loso_summary = {
    'model_version': FORECAST_MODEL_VERSION,
    'ae_model_version': AE_MODEL_VERSION,
    'protocol': (
        'outer WESAD LOSO; fold-specific frozen AE; training-only p95; '
        'inner participant LOSO alpha selection; direct +5/+10 score forecast'
    ),
    'raw_feature_windows_overlap_percent': 0,
    'ae_forecast_block_minutes': BLOCK_MINUTES,
    'ae_forecast_blocks_overlap_percent': 0,
    'rolling_origin_note': (
        'origins within one participant can share earlier distinct blocks; '
        'participant grouping, equal weighting, macro metrics, and participant '
        'bootstrap prevent treating those origins as independent people'
    ),
    'requested_subjects': WESAD_SUBJECTS,
    'evaluated_subjects': evaluated_subjects,
    'not_evaluable_subjects': not_evaluable,
    'per_subject': loso_results,
    'macro_subject_primary': macro_metrics,
    'pooled_origins_secondary': pooled_metrics,
    'deployment_gate': deployment_gate,
}

with open(
    os.path.join(FORECAST_RESULTS_DIR, 'wesad_loso_forecast_metrics.json'),
    'w',
) as file:
    json.dump(loso_summary, file, indent=2)

print('=' * 60)
print(f'INTERNAL PREDICTIVE GATE: {deployment_gate["status"]}')
print('=' * 60)
for check_name, passed in predictive_gate_checks.items():
    print(f'  {"OK" if passed else "NOT MET"}: {check_name}')
print('Participant bootstrap intervals are uncertainty evidence, not a save gate:')
for check_name, passed in uncertainty_checks.items():
    print(f'  {"ABOVE ZERO" if passed else "INCLUDES ZERO"}: {check_name}')
print(
    f'Participant bootstrap improvement CI +5: '
    f'[{ci_5["lower_95"]:.6f}, {ci_5["upper_95"]:.6f}]'
)
print(
    f'Participant bootstrap improvement CI +10: '
    f'[{ci_10["lower_95"]:.6f}, {ci_10["upper_95"]:.6f}]'
)


INTERNAL PREDICTIVE GATE: APPROVED_FOR_EXTERNAL_VALIDATION
  OK: at_least_10_evaluable_subjects
  OK: positive_macro_improvement_plus_5m
  OK: positive_macro_improvement_plus_10m
  OK: majority_of_subjects_better_at_both_horizons
Participant bootstrap intervals are uncertainty evidence, not a save gate:
  INCLUDES ZERO: participant_bootstrap_95_lower_bound_above_zero_plus_5m
  INCLUDES ZERO: participant_bootstrap_95_lower_bound_above_zero_plus_10m
Participant bootstrap improvement CI +5: [-0.005936, 0.036407]
Participant bootstrap improvement CI +10: [-0.007778, 0.062338]


In [7]:
# ============================================================
# CELL 7 - Fit the All-WESAD Model in Memory After WESAD Approval
# ============================================================

final_candidate = None
final_fitted = None
final_alpha_audit = None
final_training_counts = None
final_training_origins = 0
final_selected_alpha = None
final_p95 = float(ae_metadata['baseline_p95_raw_error'])

if passes_internal_gate:
    print('=' * 60)
    print('WESAD VALIDATION APPROVED - FITTING THE ALL-WESAD MODEL IN MEMORY')
    print('=' * 60)

    final_ae = load_frozen_ae(os.path.join(AE_MODELS_DIR, 'LSTM_AE_FINAL.pth'))
    final_data = {
        sid: wesad_subject_samples(sid, final_ae, final_p95)
        for sid in WESAD_SUBJECTS
    }
    final_selected_alpha, final_alpha_audit = select_alpha_inner_loso(
        final_data,
        WESAD_SUBJECTS,
    )
    X_final, y_final, final_subject_ids = combine_subject_data(
        final_data,
        WESAD_SUBJECTS,
    )
    final_fitted = fit_direct_ridge(
        X_final,
        y_final,
        final_subject_ids,
        final_selected_alpha,
    )
    final_candidate = fitted_ridge_to_torch(final_fitted).to(device)
    final_candidate.eval()
    final_training_counts = {
        sid: int(len(final_data[sid]['X'])) for sid in WESAD_SUBJECTS
    }
    final_training_origins = int(len(X_final))

    # Exact implementation equivalence check before the model can be saved.
    sklearn_check = predict_direct_ridge(final_fitted, X_final[:20])
    with torch.no_grad():
        torch_check = final_candidate(
            torch.tensor(X_final[:20], dtype=torch.float32, device=device)
        ).cpu().numpy()
    assert np.allclose(sklearn_check, torch_check, rtol=1e-5, atol=1e-6)

    print(f'Final alpha selected by participant LOSO: {final_selected_alpha:g}')
    print(f'All-WESAD training origins: {final_training_origins}')
    print('The fitted model remains in memory until external validation is complete.')
else:
    print('=' * 60)
    print('WESAD VALIDATION NOT APPROVED')
    print('Final fitting is skipped because the required held-out checks were not met.')
    print('=' * 60)


WESAD VALIDATION APPROVED - FITTING THE ALL-WESAD MODEL IN MEMORY
Final alpha selected by participant LOSO: 100
All-WESAD training origins: 43
The fitted model remains in memory until external validation is complete.


In [8]:
# ============================================================
# CELL 8 - Continuous AffectiveROAD Timeline and Untouched External Test
# ============================================================

# The binary EDA correctly removes middle subjective scores because they cannot be
# called baseline or stress with confidence. This forecast target is different: it
# is the future C1 physiological anomaly score, so no binary label is required.
# We therefore rebuild forecast-only timelines from every quality-valid raw minute.
# The EDA outputs and their labels remain unchanged.


def load_e4_single(path):
    """Exact E4 CSV reader used by the corrected AffectiveROAD EDA."""
    with open(path) as file:
        lines = file.read().strip().split('\n')
    start = float(lines[0].split(',')[0])
    sample_rate = float(lines[1].split(',')[0])
    data = np.asarray([
        list(map(float, line.split(','))) for line in lines[2:]
    ])
    if data.ndim == 2 and data.shape[1] == 1:
        data = data.flatten()
    return start, sample_rate, data


def filter_ar_acceleration(acc, sample_rate):
    acc_g = np.asarray(acc, dtype=np.float64) / 64.0
    b, a = signal.butter(
        4,
        0.5 / (0.5 * sample_rate),
        btype='high',
    )
    filtered = np.column_stack([
        signal.filtfilt(b, a, acc_g[:, axis]) for axis in range(3)
    ])
    return np.sqrt(np.sum(filtered ** 2, axis=1))


def load_ar_e4_signals(drive_n, drive_id):
    folder = os.path.join(AROAD_E4_DIR, f'{drive_n}-E4-{drive_id}', 'Left')
    _, bvp_fs, bvp = load_e4_single(os.path.join(folder, 'BVP.csv'))
    _, temp_fs, temp = load_e4_single(os.path.join(folder, 'TEMP.csv'))
    _, acc_fs, acc = load_e4_single(os.path.join(folder, 'ACC.csv'))
    assert bvp_fs == 64 and temp_fs == 4 and acc_fs == 32
    temp[:4] = np.nan
    return {
        'bvp': bvp,
        'temp': temp,
        'acc': filter_ar_acceleration(acc, acc_fs),
    }


def load_ar_e4_signals_drv2():
    folder1 = os.path.join(AROAD_E4_DIR, '2-E4-Drv2', 'Left1')
    folder2 = os.path.join(AROAD_E4_DIR, '2-E4-Drv2', 'Left2')

    _, bvp_fs1, bvp1 = load_e4_single(os.path.join(folder1, 'BVP.csv'))
    _, temp_fs1, temp1 = load_e4_single(os.path.join(folder1, 'TEMP.csv'))
    _, acc_fs1, acc1 = load_e4_single(os.path.join(folder1, 'ACC.csv'))
    _, bvp_fs2, bvp2 = load_e4_single(os.path.join(folder2, 'BVP.csv'))
    _, temp_fs2, temp2 = load_e4_single(os.path.join(folder2, 'TEMP.csv'))
    _, acc_fs2, acc2 = load_e4_single(os.path.join(folder2, 'ACC.csv'))
    assert (bvp_fs1, temp_fs1, acc_fs1) == (64, 4, 32)
    assert (bvp_fs2, temp_fs2, acc_fs2) == (64, 4, 32)

    temp1[:4] = np.nan
    temp2[:4] = np.nan
    left1 = {
        'bvp': bvp1,
        'temp': temp1,
        'acc': filter_ar_acceleration(acc1, acc_fs1),
    }
    left2 = {
        'bvp': bvp2,
        'temp': temp2,
        'acc': filter_ar_acceleration(acc2, acc_fs2),
    }
    # Exact split offset supplied by the corrected AffectiveROAD EDA.
    left2_4hz_offset = 14444
    return left1, left2, left2_4hz_offset


def extract_ar_hrv(bvp_window, sample_rate=64):
    """Exact accepted PPG HRV quality logic for one AffectiveROAD minute."""
    try:
        bvp_clean = nk.ppg_clean(bvp_window, sampling_rate=sample_rate)
        peak_info = nk.ppg_findpeaks(bvp_clean, sampling_rate=sample_rate)
        peak_idx = peak_info['PPG_Peaks']
    except Exception:
        return None

    if len(peak_idx) < 31:
        return None
    rr_ms = np.diff(peak_idx) / sample_rate * 1000
    rr_ms = rr_ms[(rr_ms >= 300) & (rr_ms <= 2000)]
    if len(rr_ms) < 30:
        return None

    local_median = np.median(rr_ms)
    rr_clean = rr_ms[np.abs(rr_ms - local_median) / local_median <= 0.20]
    if len(rr_clean) < 30:
        return None

    mean_rr = float(np.mean(rr_clean))
    return np.asarray([
        60000.0 / mean_rr,
        mean_rr,
        np.std(rr_clean, ddof=1),
        np.sqrt(np.mean(np.diff(rr_clean) ** 2)),
    ], dtype=np.float64)


def extract_ar_window_features(bvp_window, temp_window, acc_window, br_window):
    hrv = extract_ar_hrv(bvp_window)
    if hrv is None:
        return None

    valid_br = br_window[(br_window >= 6) & (br_window <= 40)]
    if len(br_window) == 0 or len(valid_br) / len(br_window) < 0.80:
        return None

    valid_temp = temp_window[(temp_window >= 25) & (temp_window <= 40)]
    if len(temp_window) == 0 or len(valid_temp) / len(temp_window) < 0.80:
        return None

    features = np.asarray([
        hrv[0],
        hrv[1],
        hrv[2],
        hrv[3],
        np.mean(valid_br),
        np.std(valid_br, ddof=1) if len(valid_br) > 1 else 0.0,
        np.mean(valid_temp),
        np.std(valid_temp, ddof=1) if len(valid_temp) > 1 else 0.0,
        np.mean(acc_window),
        np.std(acc_window, ddof=1),
    ], dtype=np.float64)
    return features if np.isfinite(features).all() else None


def extract_ar_recording_minutes(
    e4_signals,
    bio_df,
    local_start_4hz,
    local_end_4hz,
    bio_start_1hz,
    bio_end_1hz,
    global_start_4hz,
    recording_id,
):
    """Extract quality-valid non-overlapping minutes from one recording."""
    duration_seconds = min(
        (local_end_4hz - local_start_4hz) / 4,
        bio_end_1hz - bio_start_1hz,
    )
    if duration_seconds < WINDOW_SECONDS:
        return [], [], [], 0

    n_windows = int((duration_seconds - WINDOW_SECONDS) // STEP_SECONDS) + 1
    local_start_seconds = local_start_4hz / 4
    features, starts, recordings = [], [], []
    rejected = 0

    for window_index in range(n_windows):
        local_window_start = local_start_seconds + window_index * STEP_SECONDS
        bvp_start = int(local_window_start * 64)
        acc_start = int(local_window_start * 32)
        temp_start = int(local_window_start * 4)
        br_start = int(bio_start_1hz) + window_index * STEP_SECONDS

        bvp_window = e4_signals['bvp'][bvp_start:bvp_start + 60 * 64]
        acc_window = e4_signals['acc'][acc_start:acc_start + 60 * 32]
        temp_window = e4_signals['temp'][temp_start:temp_start + 60 * 4]
        br_window = bio_df['BR'].values[br_start:br_start + 60]

        if not (
            len(bvp_window) == 60 * 64
            and len(acc_window) == 60 * 32
            and len(temp_window) == 60 * 4
            and len(br_window) == 60
        ):
            rejected += 1
            continue

        if np.any(np.isnan(temp_window)):
            temp_window = pd.Series(temp_window).interpolate(
                limit_direction='both'
            ).bfill().ffill().to_numpy()

        feature_vector = extract_ar_window_features(
            bvp_window,
            temp_window,
            acc_window,
            br_window,
        )
        if feature_vector is None:
            rejected += 1
            continue

        features.append(feature_vector)
        starts.append(global_start_4hz / 4 + window_index * STEP_SECONDS)
        recordings.append(recording_id)

    return features, starts, recordings, rejected


ann_e4 = pd.read_csv(os.path.join(AROAD_E4_DIR, 'Annot_E4_Left.csv'))
ann_bio = pd.read_csv(os.path.join(AROAD_BIO_DIR, 'Annot_Bioharness.csv'))


class DriveNotEvaluableError(ValueError):
    """A drive cannot be scored causally with the available quality-valid data."""


def prepare_ar_forecast_timeline(drive_id):
    drive_n = int(drive_id.replace('Drv', ''))
    if drive_id == 'Drv2':
        e4_left1, e4_left2, left2_offset = load_ar_e4_signals_drv2()
    else:
        e4 = load_ar_e4_signals(drive_n, drive_id)

    bio = pd.read_csv(
        os.path.join(AROAD_BIO_DIR, f'Bio_{drive_id}.csv'),
        sep=';',
    )
    row_e4 = ann_e4[ann_e4['Drive-id'] == drive_id].iloc[0]
    row_bio = ann_bio[ann_bio['Drive_id'] == drive_id].iloc[0]

    if drive_id == 'Drv2':
        recording_ranges = [
            (
                'Left1_rest',
                e4_left1,
                int(row_e4['Rest_Start']),
                int(row_e4['Rest_End']),
                int(row_e4['Rest_Start']),
                int(row_e4['Rest_End']),
                int(row_bio['Rest_Start']),
                int(row_bio['Rest_End']),
            ),
            (
                'Left2_drive',
                e4_left2,
                int(row_e4['Z_Start']) - left2_offset,
                int(row_e4['Rest_End.1']) - left2_offset,
                int(row_e4['Z_Start']),
                int(row_e4['Rest_End.1']),
                int(row_bio['Z_Start']),
                int(row_bio['Rest_End.1']),
            ),
        ]
    else:
        recording_ranges = [
            (
                'continuous_drive',
                e4,
                int(row_e4['Rest_Start']),
                int(row_e4['Rest_End.1']),
                int(row_e4['Rest_Start']),
                int(row_e4['Rest_End.1']),
                int(row_bio['Rest_Start']),
                int(row_bio['Rest_End.1']),
            ),
        ]

    raw_features, starts, recording_ids = [], [], []
    rejected_total = 0
    for (
        recording_id,
        e4_signals,
        local_start,
        local_end,
        global_start,
        global_end,
        bio_start,
        bio_end,
    ) in recording_ranges:
        assert local_end > local_start and global_end > global_start
        feature_list, start_list, recording_list, rejected = extract_ar_recording_minutes(
            e4_signals=e4_signals,
            bio_df=bio,
            local_start_4hz=local_start,
            local_end_4hz=local_end,
            bio_start_1hz=bio_start,
            bio_end_1hz=bio_end,
            global_start_4hz=global_start,
            recording_id=recording_id,
        )
        raw_features.extend(feature_list)
        starts.extend(start_list)
        recording_ids.extend(recording_list)
        rejected_total += rejected

    raw_features = np.asarray(raw_features, dtype=np.float64)
    starts = np.asarray(starts, dtype=np.float64)
    recording_ids = np.asarray(recording_ids)
    if len(raw_features) == 0:
        raise DriveNotEvaluableError('no quality-valid physiological minutes')
    assert raw_features.ndim == 2 and raw_features.shape[1] == N_FEATURES
    assert len(np.unique(starts)) == len(starts)

    # Use only the first pre-drive rest. This is available before forecasting and
    # avoids using low-score or future road windows to define the person's baseline.
    rest_start_sec = float(row_e4['Rest_Start']) / 4.0
    rest_end_sec = float(row_e4['Rest_End']) / 4.0
    rest_out_mask = (
        (starts >= rest_start_sec - 1e-6)
        & (starts + WINDOW_SECONDS <= rest_end_sec + 1e-6)
    )
    n_rest_out = int(np.sum(rest_out_mask))
    if n_rest_out < 5:
        raise DriveNotEvaluableError(
            f'only {n_rest_out} quality-valid Rest_out baseline minutes; '
            'at least 5 are required for causal per-drive normalization'
        )
    baseline_mean = np.mean(raw_features[rest_out_mask], axis=0)
    baseline_std = np.std(raw_features[rest_out_mask], axis=0)
    baseline_std = np.where(baseline_std == 0, 1e-8, baseline_std)
    normalized = (raw_features - baseline_mean) / baseline_std
    assert np.isfinite(normalized).all()

    output_path = os.path.join(
        FORECAST_CHECKPOINT_DIR,
        f'AR_{drive_id}_continuous_forecast_timeline.npz',
    )
    np.savez_compressed(
        output_path,
        features=normalized.astype(np.float32),
        start_sec=starts,
        recording_ids=recording_ids,
        baseline_mean=baseline_mean.astype(np.float32),
        baseline_std=baseline_std.astype(np.float32),
        normalization_source=np.asarray('Rest_out_only_v1'),
        timeline_contract=np.asarray('continuous_recording_v2'),
    )
    return output_path, len(normalized), rejected_total


print('=' * 60)
print('BUILDING FORECAST-ONLY CONTINUOUS AFFECTIVEROAD TIMELINES')
print('No binary labels are assigned and the EDA outputs are unchanged.')
print('Drives without a usable pre-drive baseline are recorded and skipped safely.')
print('=' * 60)

ar_timeline_summary = {}
for drive_id in AROAD_DRIVES:
    timeline_path = os.path.join(
        FORECAST_CHECKPOINT_DIR,
        f'AR_{drive_id}_continuous_forecast_timeline.npz',
    )
    reuse_timeline = False
    if os.path.exists(timeline_path) and not REBUILD_AR_TIMELINES:
        try:
            timeline = np.load(timeline_path, allow_pickle=True)
            required_timeline_fields = {
                'features', 'start_sec', 'recording_ids',
                'normalization_source', 'timeline_contract',
            }
            reuse_timeline = (
                required_timeline_fields.issubset(timeline.files)
                and str(timeline['normalization_source'].item()) == 'Rest_out_only_v1'
                and str(timeline['timeline_contract'].item()) == 'continuous_recording_v2'
                and timeline['features'].ndim == 2
                and timeline['features'].shape[1] == N_FEATURES
                and len(timeline['features']) == len(timeline['start_sec'])
                and len(timeline['features']) == len(timeline['recording_ids'])
                and np.isfinite(timeline['features']).all()
                and np.isfinite(timeline['start_sec']).all()
            )
        except (OSError, ValueError, EOFError, KeyError):
            reuse_timeline = False

    try:
        if reuse_timeline:
            n_minutes = len(timeline['features'])
            rejected = None
            source = 'reused forecast-only timeline'
        else:
            timeline_path, n_minutes, rejected = prepare_ar_forecast_timeline(drive_id)
            source = 'rebuilt from raw AffectiveROAD'
    except DriveNotEvaluableError as error:
        ar_timeline_summary[drive_id] = {
            'status': 'not_evaluable',
            'reason': str(error),
            'file': None,
        }
        print(f'{drive_id}: not evaluable; {error}')
        continue

    ar_timeline_summary[drive_id] = {
        'status': 'ready',
        'valid_minutes': int(n_minutes),
        'rejected_minutes': int(rejected) if rejected is not None else None,
        'source': source,
        'file': timeline_path,
    }
    print(f'{drive_id}: valid minutes={n_minutes}, source={source}')

with open(
    os.path.join(FORECAST_RESULTS_DIR, 'affectiveroad_timeline_summary.json'),
    'w',
) as file:
    json.dump(ar_timeline_summary, file, indent=2)


external_ae = load_frozen_ae(os.path.join(AE_MODELS_DIR, 'LSTM_AE_FINAL.pth'))
external_results = {}
external_prediction_parts = []
external_target_parts = []
external_persistence_parts = []
external_drive_parts = []
external_available_origins = 0

print('=' * 60)
print('UNTOUCHED AFFECTIVEROAD EXTERNAL FORECAST TEST')
print('=' * 60)

for drive_id in AROAD_DRIVES:
    timeline_info = ar_timeline_summary[drive_id]
    if timeline_info['status'] != 'ready':
        external_results[drive_id] = {
            'status': 'not_evaluable',
            'reason': timeline_info['reason'],
            'n_origins': 0,
        }
        print(f'{drive_id}: excluded from external evaluation; {timeline_info["reason"]}')
        continue

    timeline_path = os.path.join(
        FORECAST_CHECKPOINT_DIR,
        f'AR_{drive_id}_continuous_forecast_timeline.npz',
    )
    timeline = np.load(timeline_path, allow_pickle=True)
    features = timeline['features'].astype(np.float32)
    starts = timeline['start_sec'].astype(np.float64)
    recording_ids = timeline['recording_ids'].astype(str)

    drive_X_parts, drive_y_parts, drive_persistence_parts = [], [], []
    for recording_id in np.unique(recording_ids):
        recording_mask = recording_ids == recording_id
        order = np.argsort(starts[recording_mask], kind='stable')
        recording_data = timeline_to_samples(
            features[recording_mask][order],
            starts[recording_mask][order],
            external_ae,
            final_p95,
        )
        if len(recording_data['X']) > 0:
            drive_X_parts.append(recording_data['X'])
            drive_y_parts.append(recording_data['y'])
            drive_persistence_parts.append(recording_data['persistence'])

    if not drive_X_parts:
        external_results[drive_id] = {
            'status': 'not_evaluable',
            'reason': (
                f'no recording has {MINIMUM_CONTINUOUS_MINUTES} '
                'quality-valid uninterrupted minutes'
            ),
            'n_origins': 0,
        }
        print(f'{drive_id}: not evaluable; no honest 20-minute origin')
        continue

    X_drive = np.concatenate(drive_X_parts, axis=0)
    y_drive = np.concatenate(drive_y_parts, axis=0)
    persistence_drive = np.concatenate(drive_persistence_parts, axis=0)
    external_available_origins += len(X_drive)

    if final_candidate is None:
        external_results[drive_id] = {
            'status': 'data_evaluable_but_internal_model_not_approved',
            'n_origins': int(len(X_drive)),
        }
        print(f'{drive_id}: origins={len(X_drive)}, internal candidate unavailable')
        continue

    with torch.no_grad():
        predictions_drive = final_candidate(
            torch.tensor(X_drive, dtype=torch.float32, device=device)
        ).cpu().numpy()
    metrics_drive = horizon_metrics(
        predictions_drive,
        y_drive,
        persistence_drive,
    )
    metrics_drive.update({'status': 'evaluated', 'n_origins': int(len(X_drive))})
    external_results[drive_id] = metrics_drive
    external_prediction_parts.append(predictions_drive)
    external_target_parts.append(y_drive)
    external_persistence_parts.append(persistence_drive)
    external_drive_parts.append(np.repeat(drive_id, len(X_drive)))

    print(
        f'{drive_id}: origins={len(X_drive)}, '
        f'improvement +5={metrics_drive["MAE_improvement_over_persistence_plus_5m"]:.4f}, '
        f'+10={metrics_drive["MAE_improvement_over_persistence_plus_10m"]:.4f}'
    )

if external_prediction_parts:
    external_predictions = np.concatenate(external_prediction_parts, axis=0)
    external_targets = np.concatenate(external_target_parts, axis=0)
    external_persistence = np.concatenate(external_persistence_parts, axis=0)
    external_drive_ids = np.concatenate(external_drive_parts)
    external_pooled = horizon_metrics(
        external_predictions,
        external_targets,
        external_persistence,
    )

    evaluated_drive_ids = [
        drive_id for drive_id, result in external_results.items()
        if result['status'] == 'evaluated'
    ]
    evaluated_drive_results = [external_results[drive_id] for drive_id in evaluated_drive_ids]
    external_macro = {
        metric_name: float(np.mean([
            result[metric_name] for result in evaluated_drive_results
        ]))
        for metric_name in metric_names
    }
    external_better_both = int(np.sum([
        result['MAE_improvement_over_persistence_plus_5m'] > 0
        and result['MAE_improvement_over_persistence_plus_10m'] > 0
        for result in evaluated_drive_results
    ]))
    external_majority_required = len(evaluated_drive_results) // 2 + 1

    external_gate_checks = {
        'at_least_10_evaluable_drives': bool(
            len(evaluated_drive_results) >= MIN_EXTERNAL_DRIVES
        ),
        'at_least_30_external_origins': bool(
            external_available_origins >= MIN_EXTERNAL_ORIGINS
        ),
        'positive_drive_macro_improvement_plus_5m': bool(
            external_macro['MAE_improvement_over_persistence_plus_5m'] > 0
        ),
        'positive_drive_macro_improvement_plus_10m': bool(
            external_macro['MAE_improvement_over_persistence_plus_10m'] > 0
        ),
        'majority_of_drives_better_at_both_horizons': bool(
            external_better_both >= external_majority_required
        ),
    }
    external_pass = bool(all(external_gate_checks.values()))
    external_status = 'APPROVED' if external_pass else 'NOT_APPROVED'

    np.savez_compressed(
        os.path.join(FORECAST_RESULTS_DIR, 'affectiveroad_forecast_outputs.npz'),
        predictions=external_predictions.astype(np.float32),
        targets=external_targets.astype(np.float32),
        persistence=external_persistence.astype(np.float32),
        drive_ids=external_drive_ids,
        horizons_minutes=np.asarray(REPORTED_HORIZONS_MINUTES, dtype=np.int16),
    )
else:
    external_pooled = None
    external_macro = None
    external_gate_checks = {}
    external_pass = False
    external_status = (
        'NOT_RUN_WESAD_NOT_APPROVED'
        if external_available_origins > 0
        else 'NOT_EVALUABLE_NO_CONTINUOUS_RUNS'
    )
    evaluated_drive_ids = []
    external_better_both = 0
    external_majority_required = 0

external_summary = {
    'model_version': FORECAST_MODEL_VERSION,
    'protocol': (
        'untouched raw AffectiveROAD signals; every quality-valid minute retained '
        'without binary relabelling; same continuous recording only; true timestamp '
        'gaps split sequences while route annotations do not; distinct non-overlapping '
        'five-minute AE blocks; per-drive normalization fitted only on the first '
        'pre-drive Rest_out period'
    ),
    'status': external_status,
    'n_available_origins': int(external_available_origins),
    'evaluated_drives': evaluated_drive_ids,
    'per_drive': external_results,
    'macro_drive_primary': external_macro,
    'pooled_origins_secondary': external_pooled,
    'external_gate_checks': external_gate_checks,
    'drives_better_at_both_horizons': int(external_better_both),
    'majority_required': int(external_majority_required),
    'passes_external_gate': bool(external_pass),
    'note': (
        'Middle subjective-score minutes are retained only because this target is '
        'future physiological C1 score. They are never assigned a binary stress label.'
    ),
}

with open(
    os.path.join(FORECAST_RESULTS_DIR, 'affectiveroad_forecast_metrics.json'),
    'w',
) as file:
    json.dump(external_summary, file, indent=2)

del external_ae
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('\nAffectiveROAD drive-macro results')
if external_macro is not None:
    print(
        f'  +5 improvement over persistence: '
        f'{external_macro["MAE_improvement_over_persistence_plus_5m"]:.6f}'
    )
    print(
        f'  +10 improvement over persistence: '
        f'{external_macro["MAE_improvement_over_persistence_plus_10m"]:.6f}'
    )
for check_name, passed in external_gate_checks.items():
    print(f'  {"OK" if passed else "NOT MET"}: {check_name}')
print(f'AffectiveROAD external forecast status: {external_status}')


BUILDING FORECAST-ONLY CONTINUOUS AFFECTIVEROAD TIMELINES
No binary labels are assigned and the EDA outputs are unchanged.
Drives without a usable pre-drive baseline are recorded and skipped safely.
Drv1: valid minutes=59, source=reused forecast-only timeline
Drv2: not evaluable; only 0 quality-valid Rest_out baseline minutes; at least 5 are required for causal per-drive normalization
Drv3: valid minutes=83, source=rebuilt from raw AffectiveROAD
Drv4: valid minutes=86, source=rebuilt from raw AffectiveROAD
Drv5: valid minutes=92, source=rebuilt from raw AffectiveROAD
Drv6: valid minutes=83, source=rebuilt from raw AffectiveROAD
Drv7: valid minutes=82, source=rebuilt from raw AffectiveROAD
Drv8: valid minutes=86, source=rebuilt from raw AffectiveROAD
Drv9: valid minutes=77, source=rebuilt from raw AffectiveROAD
Drv10: valid minutes=90, source=rebuilt from raw AffectiveROAD
Drv11: valid minutes=88, source=rebuilt from raw AffectiveROAD
Drv12: valid minutes=77, source=rebuilt from raw Aff

In [9]:
# ============================================================
# CELL 9 - Final Save Decision and Complete Audit Trail
# ============================================================

final_model_path = None
final_metadata_path = None

# A final checkpoint requires both the WESAD predictive gate and the untouched
# continuous AffectiveROAD external gate. No internal-only production model is saved.
if passes_internal_gate and external_pass is True:
    save_status = 'EXTERNALLY_VALIDATED_FINAL'
    final_filename = 'DIRECT_RIDGE_FORECAST_FINAL.pth'
else:
    save_status = 'NOT_SAVED'
    final_filename = None

if final_filename is not None:
    assert final_candidate is not None and final_fitted is not None
    final_model_path = os.path.join(FORECAST_MODELS_DIR, final_filename)
    torch.save(final_candidate.state_dict(), final_model_path)

    model_metadata = {
        'model_version': FORECAST_MODEL_VERSION,
        'model_file': os.path.basename(final_model_path),
        'save_status': save_status,
        'model_type': 'direct_multioutput_ridge_logit_score_delta',
        'ae_model_version': AE_MODEL_VERSION,
        'ae_model_file': 'LSTM_AE_FINAL.pth',
        'c1_score_formula': ae_metadata['predict_score_formula'],
        'baseline_p95_raw_error': final_p95,
        'score_note': 'bounded monotonic anomaly score, not a probability',
        'history_blocks': HISTORY_BLOCKS,
        'history_block_minutes': BLOCK_MINUTES,
        'history_raw_minutes': HISTORY_BLOCKS * BLOCK_MINUTES,
        'forecast_horizons_minutes': REPORTED_HORIZONS_MINUTES,
        'raw_feature_window_overlap_percent': 0,
        'ae_forecast_block_overlap_percent': 0,
        'forecast_strategy': (
            'direct +5/+10 prediction of logit-score change relative to persistence'
        ),
        'recursive_decoding': False,
        'teacher_forcing': False,
        'manual_posthoc_anchoring': False,
        'training_subjects': WESAD_SUBJECTS,
        'excluded_subjects': ['S3', 'S6'],
        'training_origins_by_subject': final_training_counts,
        'n_training_origins': final_training_origins,
        'subject_weighting': 'each participant has total training weight 1',
        'selected_alpha': final_selected_alpha,
        'alpha_selection': final_alpha_audit,
        'n_linear_coefficients_and_biases': 6,
        'wesad_loso_results_file': 'wesad_loso_forecast_metrics.json',
        'deployment_gate_file': 'deployment_gate.json',
        'affectiveroad_results_file': 'affectiveroad_forecast_metrics.json',
        'external_validation_status': external_status,
        'external_evaluated_drives': evaluated_drive_ids,
        'external_not_evaluable_drives': [
            drive_id for drive_id, result in external_results.items()
            if result['status'] == 'not_evaluable'
        ],
        'endpoint_change_required': True,
        'endpoint_note': (
            'Load with DirectScoreForecaster. The model takes two C1 score history '
            'values and returns [score_plus_5m, score_plus_10m].'
        ),
        'trained_at': datetime.now(timezone.utc).isoformat(),
    }

    final_metadata_path = os.path.join(
        FORECAST_MODELS_DIR,
        os.path.splitext(final_filename)[0] + '_metadata.json',
    )
    with open(final_metadata_path, 'w') as file:
        json.dump(model_metadata, file, indent=2)

run_summary = {
    'model_version': FORECAST_MODEL_VERSION,
    'save_status': save_status,
    'saved_model': final_model_path,
    'saved_metadata': final_metadata_path,
    'internal_gate': deployment_gate,
    'external_status': external_status,
    'next_action': (
        'Update the endpoint only if a model file was saved, after reviewing these '
        'held-out results. Otherwise keep the endpoint unchanged.'
    ),
}
with open(os.path.join(FORECAST_RESULTS_DIR, 'run_summary.json'), 'w') as file:
    json.dump(run_summary, file, indent=2)

print('=' * 60)
print('FORECASTING NOTEBOOK COMPLETE')
print('=' * 60)
print(f'Save decision: {save_status}')
print(f'WESAD held-out metrics: {os.path.join(FORECAST_RESULTS_DIR, "wesad_loso_forecast_metrics.json")}')
print(f'Deployment gate: {os.path.join(FORECAST_RESULTS_DIR, "deployment_gate.json")}')
print(f'AffectiveROAD audit: {os.path.join(FORECAST_RESULTS_DIR, "affectiveroad_forecast_metrics.json")}')

if final_model_path is not None:
    print(f'Saved model: {final_model_path}')
    print(f'Metadata: {final_metadata_path}')
else:
    print('Final model not saved because every required validation check was not met.')


FORECASTING NOTEBOOK COMPLETE
Save decision: EXTERNALLY_VALIDATED_FINAL
WESAD held-out metrics: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs- New/DIRECT_FORECASTER/results/wesad_loso_forecast_metrics.json
Deployment gate: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs- New/DIRECT_FORECASTER/results/deployment_gate.json
AffectiveROAD audit: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs- New/DIRECT_FORECASTER/results/affectiveroad_forecast_metrics.json
Saved model: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs- New/DIRECT_FORECASTER/models/DIRECT_RIDGE_FORECAST_FINAL.pth
Metadata: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs- New/DIRECT_FORECASTER/models/DIRECT_RIDGE_FORECAST_FINAL_metadata.json
